# 面试题：KV Cache 如何实现，怎样验证增量解码与全量前向数值一致？

## 面试回答主线

自回归生成第 `t` 个 token 时，历史 token 的 K/V 在同一层不会改变，因此可以在 prefill 阶段计算一次并缓存，decode 阶段只投影新 token 的 Q/K/V。新 Q 仍需与全部历史 K 做点积，所以单步 attention 是线性增长，但避免了反复投影历史 token 和重算完整 `L×L` 矩阵。正确性标准不是“速度看起来更快”，而是逐步 cached 输出与同模型全量 causal 输出在容差内一致。缓存还必须绑定 token 前缀、模型/adapter 版本、位置策略和层；off-by-one position 会产生无异常但错误的 logits。

## 真实案例：六个客服回复生成请求

每条记录给出已经到达服务端的 prompt token 与随后教师强制的生成 token，用于逐步对比全量和 cache 路径。数据为教学事件流，`generated` 不是模型质量样本，只用于数值一致性与计算量核算。

In [1]:
import math  # 导入平方根以实现 scaled dot-product attention。
import hashlib  # 导入哈希函数以构造可审计的缓存键。
import torch  # 导入 PyTorch 以手写全量与增量注意力。
from torch import nn  # 导入基础模块和可学习参数抽象。
torch.set_num_threads(1)  # 小张量教学实验固定单线程以保证快速复现。
requests = [  # 构造六个包含 prompt 与生成 token 的客服请求。
    {"request_id": "R-01", "prompt": ["[BOS]", "订单", "查询"], "generated": ["正在", "处理中", "[EOS]"]},  # 订单查询生成三 token 回复。
    {"request_id": "R-02", "prompt": ["[BOS]", "退款", "进度", "查询"], "generated": ["预计", "明天", "到账", "[EOS]"]},  # 退款请求拥有更长 prompt 与回复。
    {"request_id": "R-03", "prompt": ["[BOS]", "账号", "无法", "登录"], "generated": ["请", "重置", "密码", "[EOS]"]},  # 登录故障回复四 token。
    {"request_id": "R-04", "prompt": ["[BOS]", "包裹", "未", "送达"], "generated": ["正在", "联系", "物流", "[EOS]"]},  # 物流场景复用正在 token。
    {"request_id": "R-05", "prompt": ["[BOS]", "优惠券", "失效"], "generated": ["请", "检查", "有效期", "[EOS]"]},  # 优惠券场景使用三长度 prompt。
    {"request_id": "R-06", "prompt": ["[BOS]", "重复", "扣款", "申诉"], "generated": ["已经", "升级", "专员", "[EOS]"]},  # 资金风险回复四 token。
]  # 结束生成请求列表。
print("请求    prompt tokens                         generated tokens")  # 输出真实案例输入表标题。
for item in requests:  # 逐条展示请求前缀和后续 token。
    print(f"{item['request_id']:<7} {' '.join(item['prompt']):<36} {' '.join(item['generated'])}")  # 输出可读的生成事件流。

请求    prompt tokens                         generated tokens
R-01    [BOS] 订单 查询                          正在 处理中 [EOS]
R-02    [BOS] 退款 进度 查询                       预计 明天 到账 [EOS]
R-03    [BOS] 账号 无法 登录                       请 重置 密码 [EOS]
R-04    [BOS] 包裹 未 送达                        正在 联系 物流 [EOS]
R-05    [BOS] 优惠券 失效                         请 检查 有效期 [EOS]
R-06    [BOS] 重复 扣款 申诉                       已经 升级 专员 [EOS]


## Baseline（基线）：每步重新计算完整前缀

朴素路径在生成每个新 token 后，把 `prompt + 已生成` 全部再次送入 causal attention。它是数值参考，但历史 token 的 Q/K/V 投影和历史两两 attention 被重复计算。先按真实请求长度计算投影 token 行数与 score 元素数。

In [2]:
def full_recompute_work(prompt_length, generated_length):  # 计算逐步全量前向的教学计算量代理。
    lengths = [prompt_length + step for step in range(1, generated_length + 1)]  # 枚举每次加入新 token 后的完整前缀长度。
    projected_rows = sum(lengths)  # 全量路径每步都会重新投影所有前缀 token。
    score_elements = sum(length * length for length in lengths)  # 全量路径每步都会形成完整方形 attention 分数矩阵。
    return projected_rows, score_elements  # 返回两类可直接对比的工作量。
def cached_work(prompt_length, generated_length):  # 计算 prefill 加逐 token cache decode 的工作量代理。
    projected_rows = prompt_length + generated_length  # 每个 prompt 或生成 token 只投影一次。
    score_elements = prompt_length * prompt_length + sum(prompt_length + step for step in range(1, generated_length + 1))  # prefill 为方阵，decode 每步只有一行 QK。
    return projected_rows, score_elements  # 返回与基线同口径的工作量。
print("请求    全量投影/cache投影  全量score/cache score  score减少")  # 输出逐请求计算量对照表标题。
work_summary = {}  # 保存各请求工作量供后续结果表使用。
for item in requests:  # 按真实 prompt 和生成长度计算代理量。
    full_rows, full_scores = full_recompute_work(len(item["prompt"]), len(item["generated"]))  # 计算朴素全量工作量。
    cache_rows, cache_scores = cached_work(len(item["prompt"]), len(item["generated"]))  # 计算 KV cache 工作量。
    work_summary[item["request_id"]] = (full_rows, cache_rows, full_scores, cache_scores)  # 保存当前请求的两组指标。
    reduction = 1.0 - cache_scores / full_scores  # 计算 attention score 元素的相对减少比例。
    print(f"{item['request_id']:<7} {full_rows:>6}/{cache_rows:<6} {full_scores:>10}/{cache_scores:<10} {reduction:>9.1%}")  # 输出同请求下的计算对照。

请求    全量投影/cache投影  全量score/cache score  score减少
R-01        15/6              77/24             68.8%
R-02        26/8             174/42             75.9%
R-03        26/8             174/42             75.9%
R-04        26/8             174/42             75.9%
R-05        22/7             126/31             75.4%
R-06        26/8             174/42             75.9%


## 核心实现：同一组参数的 full、prefill 与 decode_step

`forward_full` 创建 causal 方阵；`prefill` 返回最后输出和分层 cache；`decode_step` 只投影一个新 token，把新 K/V 追加到缓存，并用单行 Q 查询全部历史。位置 embedding 根据 `cache['length']` 续接。

In [3]:
class TinyCachedAttention(nn.Module):  # 定义同时支持全量与 KV cache 增量路径的最小注意力层。
    def __init__(self, vocabulary_size, model_dim, max_length):  # 初始化 token/position 表和 Q/K/V/O 参数。
        super().__init__()  # 注册基础模块状态以追踪参数。
        self.model_dim = model_dim  # 保存隐藏维度供点积缩放。
        self.token_weight = nn.Parameter(torch.randn(vocabulary_size, model_dim) * 0.15)  # 创建固定实验使用的 token embedding 表。
        self.position_weight = nn.Parameter(torch.randn(max_length, model_dim) * 0.08)  # 创建绝对位置 embedding 表。
        self.query_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.12)  # 创建查询投影矩阵。
        self.key_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.12)  # 创建键投影矩阵。
        self.value_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.12)  # 创建值投影矩阵。
        self.output_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.12)  # 创建 attention 输出投影矩阵。
    def embed(self, ids, positions):  # 按显式 position id 组合 token 与位置表示。
        return self.token_weight[ids] + self.position_weight[positions]  # 返回可直接投影为 Q/K/V 的隐藏状态。
    def project(self, hidden):  # 使用同一组参数投影 Q、K、V。
        queries = hidden @ self.query_weight  # 计算查询张量。
        keys = hidden @ self.key_weight  # 计算键张量。
        values = hidden @ self.value_weight  # 计算值张量。
        return queries, keys, values  # 返回三组投影供 full 与 cache 路径复用。
    def forward_full(self, ids):  # 对完整前缀计算标准 causal self-attention。
        length = ids.shape[0]  # 读取当前完整序列长度。
        positions = torch.arange(length, dtype=torch.long)  # 为全部 token 创建从零递增的绝对位置。
        hidden = self.embed(ids, positions)  # 组合 token 与位置表示。
        queries, keys, values = self.project(hidden)  # 一次投影完整前缀的 Q/K/V。
        scores = queries @ keys.transpose(0, 1) / math.sqrt(self.model_dim)  # 创建完整长度乘长度的缩放点积分数。
        future = torch.triu(torch.ones(length, length, dtype=torch.bool), diagonal=1)  # 创建严格上三角未来位置 mask。
        scores = scores.masked_fill(future, torch.finfo(scores.dtype).min)  # 阻止每个位置读取任何未来 token。
        attention = torch.softmax(scores, dim=-1)  # 沿 key 维归一化每行分数。
        outputs = attention @ values @ self.output_weight  # 加权 V 并执行输出投影。
        return outputs, attention, {"keys": keys, "values": values, "length": length}  # 返回全位置输出、权重和可复用缓存。
    def prefill(self, prompt_ids):  # 对 prompt 执行一次全量前向并建立初始 KV cache。
        outputs, attention, cache = self.forward_full(prompt_ids)  # 复用标准 causal 路径计算 prompt。
        return outputs, attention, cache  # 返回 prompt 输出和包含全部历史 K/V 的缓存。
    def decode_step(self, token_id, cache, position_override=None):  # 只为一个新 token 计算并追加 K/V。
        position = cache["length"] if position_override is None else position_override  # 正常使用缓存长度，测试时允许注入错误位置。
        ids = token_id.reshape(1)  # 把标量新 token id 整理为一长度序列。
        positions = torch.tensor([position], dtype=torch.long)  # 创建新 token 的唯一绝对位置 id。
        hidden = self.embed(ids, positions)  # 组合新 token 与正确或故意错误的位置表示。
        query, new_key, new_value = self.project(hidden)  # 只投影当前一个 token 的 Q/K/V。
        keys = torch.cat([cache["keys"], new_key], dim=0)  # 把新 K 追加到历史键缓存。
        values = torch.cat([cache["values"], new_value], dim=0)  # 把新 V 追加到历史值缓存。
        scores = query @ keys.transpose(0, 1) / math.sqrt(self.model_dim)  # 用单行 Q 查询全部历史和当前 K。
        attention = torch.softmax(scores, dim=-1)  # 对当前可见前缀归一化注意力权重。
        output = attention @ values @ self.output_weight  # 生成当前 token 的上下文化输出。
        updated_cache = {"keys": keys, "values": values, "length": cache["length"] + 1}  # 创建长度递增的新缓存状态。
        return output, attention, updated_cache  # 返回单步输出、单行权重和更新后缓存。
all_tokens = sorted({token for item in requests for token in item["prompt"] + item["generated"]})  # 收集六个请求出现的完整 token 集合。
token_to_id = {token: index for index, token in enumerate(all_tokens)}  # 创建稳定的 token id 映射。
torch.manual_seed(89)  # 固定注意力层全部参数以保证数值结果可复现。
cached_attention = TinyCachedAttention(len(all_tokens), 12, 20)  # 创建十二维单头 cache 教学模型。
focus_prompt_ids = torch.tensor([token_to_id[token] for token in requests[0]["prompt"]], dtype=torch.long)  # 转换焦点请求 prompt。
focus_outputs, focus_attention, focus_cache = cached_attention.prefill(focus_prompt_ids)  # 对焦点 prompt 建立真实 KV cache。
print("prefill K/V 形状：", focus_cache["keys"].shape, focus_cache["values"].shape)  # 输出缓存的 token 行数和隐藏维度。
print("prefill 最后一行 attention：", [round(float(value), 4) for value in focus_attention[-1]])  # 输出 prompt 末位置如何读取历史。

prefill K/V 形状： torch.Size([3, 12]) torch.Size([3, 12])
prefill 最后一行 attention： [0.3326, 0.3332, 0.3342]


## 数值一致性：六请求、每个生成 step 都对比

每加入一个教师 token，参考路径重新计算完整前缀并取最后一行；cache 路径在同一模型上执行一次 `decode_step`。下面输出逐请求最大绝对误差、最终缓存形状和前述工作量。

In [4]:
consistency_rows = []  # 创建列表保存每个请求的逐步最大误差与缓存状态。
all_step_errors = []  # 收集所有生成 step 的误差供全局回归测试。
for item in requests:  # 逐个请求独立建立和推进 KV cache。
    prompt_ids = torch.tensor([token_to_id[token] for token in item["prompt"]], dtype=torch.long)  # 转换当前 prompt token id。
    generated_ids = [torch.tensor(token_to_id[token], dtype=torch.long) for token in item["generated"]]  # 转换每个教师生成 token id。
    with torch.no_grad():  # 一致性评估不需要梯度图。
        cached_outputs, cached_weights, cache = cached_attention.prefill(prompt_ids)  # 对当前 prompt 执行一次 prefill。
        current_ids = prompt_ids.clone()  # 保存参考全量路径当前已见 token。
        request_errors = []  # 保存当前请求每一步 cached 与 full 的差异。
        for generated_id in generated_ids:  # 按真实生成顺序逐 token 推进。
            current_ids = torch.cat([current_ids, generated_id.reshape(1)], dim=0)  # 把新 token 加入参考完整前缀。
            full_outputs, full_weights, full_cache = cached_attention.forward_full(current_ids)  # 重新计算朴素全量 causal 参考。
            cached_step, cached_step_weights, cache = cached_attention.decode_step(generated_id, cache)  # 仅投影新 token 并追加缓存。
            step_error = float((full_outputs[-1] - cached_step[0]).abs().max())  # 比较同一新 token 的上下文化输出。
            request_errors.append(step_error)  # 保存当前 step 数值误差。
            all_step_errors.append(step_error)  # 保存到跨请求误差集合。
    work = work_summary[item["request_id"]]  # 读取当前请求的全量与 cache 工作量代理。
    consistency_rows.append((item["request_id"], max(request_errors), cache["keys"].shape[0], work))  # 保存最大误差、最终长度和工作量。
print("请求    最大|full-cache|  最终KV长度  投影行 full/cache  score full/cache")  # 输出逐请求一致性结果表标题。
for request_id, max_error, cache_length, work in consistency_rows:  # 遍历并展示所有请求结果。
    print(f"{request_id:<7} {max_error:>16.9f} {cache_length:>11} {work[0]:>8}/{work[1]:<5} {work[2]:>8}/{work[3]:<6}")  # 输出误差、缓存长度和计算量。
print(f"全部 {len(all_step_errors)} 个 decode step 的全局最大误差：{max(all_step_errors):.9f}")  # 输出最严格的跨请求一致性指标。

请求    最大|full-cache|  最终KV长度  投影行 full/cache  score full/cache
R-01         0.000000004           6       15/6           77/24    
R-02         0.000000004           8       26/8          174/42    
R-03         0.000000006           8       26/8          174/42    
R-04         0.000000006           8       26/8          174/42    
R-05         0.000000004           7       22/7          126/31    
R-06         0.000000006           8       26/8          174/42    
全部 23 个 decode step 的全局最大误差：0.000000006


## 结果解读

cache 路径与 full 路径使用相同 token/position 表和 Q/K/V/O 参数，因此误差只来自浮点运算顺序，应该接近机器精度。投影行从“每步完整前缀求和”降为“每个 token 一次”，score 矩阵也从多次方阵变成 prefill 方阵加若干单行。KV cache 以显存换计算，长上下文下缓存容量会成为新的瓶颈。

## 失败案例：decode position off-by-one

prefill 三个 token 后，新 token 应使用位置 3。下面故意传入位置 2：代码不会报错，K/V 长度也正常增加，但输出与全量参考明显不同；恢复 `position=cache['length']` 后重新对齐。

In [5]:
failure_item = requests[0]  # 选择第一条请求复现无异常的 position off-by-one。
failure_prompt_ids = torch.tensor([token_to_id[token] for token in failure_item["prompt"]], dtype=torch.long)  # 转换三 token prompt。
failure_next_id = torch.tensor(token_to_id[failure_item["generated"][0]], dtype=torch.long)  # 选择第一个教师生成 token。
with torch.no_grad():  # 错误与修复对照不需要梯度图。
    _, _, failure_cache = cached_attention.prefill(failure_prompt_ids)  # 建立长度三的正确 prompt cache。
    reference_ids = torch.cat([failure_prompt_ids, failure_next_id.reshape(1)], dim=0)  # 构造包含新 token 的全量参考序列。
    reference_output = cached_attention.forward_full(reference_ids)[0][-1]  # 取得全量 causal 路径的新 token 输出。
    wrong_output = cached_attention.decode_step(failure_next_id, failure_cache, position_override=failure_cache["length"] - 1)[0][0]  # 故意复用最后一个 prompt 的位置二。
    fixed_output = cached_attention.decode_step(failure_next_id, failure_cache)[0][0]  # 使用 cache length 三作为正确新位置。
wrong_position_error = float((reference_output - wrong_output).abs().max())  # 量化 off-by-one 造成的静默输出漂移。
fixed_position_error = float((reference_output - fixed_output).abs().max())  # 量化修复后与全量前向的剩余误差。
def cache_key(token_ids, model_version, adapter_version, position_policy):  # 构造绑定前缀和运行配置的可审计缓存键。
    payload = "|".join([",".join(str(int(value)) for value in token_ids), model_version, adapter_version, position_policy])  # 规范化拼接所有会影响 K/V 的字段。
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]  # 返回紧凑但稳定的十六进制键摘要。
valid_key = cache_key(failure_prompt_ids, "model-v3", "adapter-refund-v2", "absolute-v1")  # 为当前 prompt 与版本生成正确缓存键。
changed_key = cache_key(failure_prompt_ids, "model-v3", "adapter-refund-v3", "absolute-v1")  # 只改变 adapter 版本生成另一个键。
print(f"错误 position={failure_cache['length'] - 1}：最大输出误差={wrong_position_error:.6f}")  # 展示不会抛异常的 off-by-one 结果。
print(f"修复 position={failure_cache['length']}：最大输出误差={fixed_position_error:.9f}")  # 展示使用缓存长度后的数值一致性。
print("缓存键绑定 adapter：", valid_key, changed_key, "，是否误复用：", valid_key == changed_key)  # 展示版本变化会强制 cache miss。

错误 position=2：最大输出误差=0.008233
修复 position=3：最大输出误差=0.000000004
缓存键绑定 adapter： 1291d5da07ec092a 0c0102e0059089e4 ，是否误复用： False


## 生产差距与落地清单

教学实现是单层、单请求、拼接式缓存；线上通常使用每层 K/V、GQA/MQA、分页 block、连续批处理与预分配内存，不能每步 `torch.cat`。缓存键和页表必须绑定模型、adapter、tokenizer、RoPE/position、前缀 token 与租户安全域；取消请求时及时释放引用。验证应覆盖 prefill chunking、不同 batch 重排、beam 复制/回滚、prefix sharing、量化 KV 和全量/增量 logits 容差，并同时监控命中率、显存碎片与 token 延迟。

## 最小回归测试

断言只保护逐步数值一致、工作量下降、位置续接和缓存键隔离；完整的六请求事件流和逐 step 比较才是学习主体。

In [6]:
assert max(all_step_errors) < 1e-6  # 验证六请求全部增量输出与全量 causal 输出近似一致。
assert all(row[2] == len(item["prompt"]) + len(item["generated"]) for row, item in zip(consistency_rows, requests))  # 验证最终 KV 行数等于已消费 token 总数。
assert all(work[1] < work[0] for work in work_summary.values())  # 验证每个请求的 cache 投影行数都少于全量重算。
assert all(work[3] < work[2] for work in work_summary.values())  # 验证每个请求的 cache score 元素都少于全量重算。
assert wrong_position_error > 1e-4  # 固化 decode position off-by-one 的静默错误案例。
assert fixed_position_error < 1e-6  # 验证使用 cache length 后恢复全量前向一致性。
assert valid_key != changed_key  # 验证 adapter 版本变化不会误复用旧 K/V cache。
print("最小回归测试通过：逐步数值一致、计算复用、位置续接和版本隔离均符合预期。")  # 输出完整顺序执行成功的明确结论。

最小回归测试通过：逐步数值一致、计算复用、位置续接和版本隔离均符合预期。
